In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
!touch utils.py

In [4]:
!pip3 install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.1 MB/s eta 0:00:00a 0:00:01


In [5]:
!pip3 install py-tgb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.8 MB/s eta 0:00:0000:01


In [6]:
!pip3 install tgb

  Preparing metadata (setup.py) ... done
  Created wheel for tgb: filename=tgb-1.2.0-py3-none-any.whl size=1509 sha256=8fe1092ac6ffe521eedf1fc24250bd483e4bb15501723d077b5da9795cbad88d
  Stored in directory: /root/.cache/pip/wheels/9d/9c/52/6db3245683241d0fe36d2336ec8a145efb4bc9ad3644309f15
Successfully built tgb


In [5]:
code = """
import numpy as np
import pandas as pd
import scipy.sparse as sp
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score, roc_auc_score

from torch_geometric.data import TemporalData
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    IdentityMessage,
    LastAggregator,
    LastNeighborLoader,
)

# ---------------------------------------------------------------------------
# Device
# ---------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Model Definitions
# ---------------------------------------------------------------------------

class GraphAttentionEmbedding(nn.Module):

    def __init__(self, in_c, out_c, msg_dim, time_enc):
        super().__init__()
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(
            in_c, out_c // 2, heads=2, dropout=0.1, edge_dim=edge_dim
        )
        self.time_enc = time_enc

    def forward(self, x, last, edge_index, t, msg):
        rel = last[edge_index[0]] - t
        te = self.time_enc(rel.to(x.dtype))
        return self.conv(x, edge_index, torch.cat([te, msg], dim=-1))

class LinkPredictor(nn.Module):

    def __init__(self, c):
        super().__init__()
        self.l1 = nn.Linear(c, c)
        self.l2 = nn.Linear(c, c)
        self.lf = nn.Linear(c, 1)

    def forward(self, a, b):
        h = self.l1(a) + self.l2(b)
        return self.lf(h.relu())

class MessageDstPredictor(nn.Module):

    def __init__(self, num_nodes, msg_dim, emb_dim=64):
        super().__init__()
        self.src_emb = nn.Embedding(num_nodes, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim + msg_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_nodes),
        )

    def forward(self, src, msg):
        return self.mlp(torch.cat([self.src_emb(src), msg], dim=-1))

# ---------------------------------------------------------------------------
# Model Factory
# ---------------------------------------------------------------------------

def create_model(num_nodes, msg_dim, lr=1e-3):
    mem_dim = time_dim = emb_dim = 100

    memory = TGNMemory(
        num_nodes, msg_dim, mem_dim, time_dim,
        message_module=IdentityMessage(msg_dim, mem_dim, time_dim),
        aggregator_module=LastAggregator(),
    ).to(device)

    gnn = GraphAttentionEmbedding(
        mem_dim, emb_dim, msg_dim, memory.time_enc
    ).to(device)

    link_pred = LinkPredictor(emb_dim).to(device)

    optimizer = torch.optim.Adam(
        list(memory.parameters()) +
        list(gnn.parameters()) +
        list(link_pred.parameters()),
        lr=lr,
    )

    assoc = torch.empty(num_nodes, dtype=torch.long, device=device)
    return memory, gnn, link_pred, optimizer, assoc

# ---------------------------------------------------------------------------
# Training & Evaluation
# ---------------------------------------------------------------------------

def train_epoch(loader, memory, gnn, link_pred, optimizer, assoc, neighbor_loader, data):
    memory.train(); gnn.train(); link_pred.train()
    memory.reset_state(); neighbor_loader.reset_state()

    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        n_id, e_idx, e_id = neighbor_loader(batch.n_id)
        assoc[n_id] = torch.arange(n_id.size(0), device=device)

        z, last = memory(n_id)
        z = gnn(z, last, e_idx, data.t[e_id], data.msg[e_id])

        pos = link_pred(z[assoc[batch.src]], z[assoc[batch.dst]])
        neg = link_pred(z[assoc[batch.src]], z[assoc[batch.neg_dst]])

        loss = criterion(pos, torch.ones_like(pos)) + \
               criterion(neg, torch.zeros_like(neg))

        loss.backward()
        optimizer.step()

        memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        neighbor_loader.insert(batch.src, batch.dst)
        memory.detach()

        total_loss += loss.item() * batch.num_events

    return total_loss / loader.data.num_events

@torch.no_grad()
def eval_epoch(loader, memory, gnn, link_pred, assoc, neighbor_loader, data):
    memory.eval(); gnn.eval(); link_pred.eval()
    aps, aucs, mrrs = [], [], []

    for batch in loader:
        batch = batch.to(device)
        n_id, e_idx, e_id = neighbor_loader(batch.n_id)
        assoc[n_id] = torch.arange(n_id.size(0), device=device)

        z, last = memory(n_id)
        z = gnn(z, last, e_idx, data.t[e_id], data.msg[e_id])

        pos = link_pred(z[assoc[batch.src]], z[assoc[batch.dst]])
        neg = link_pred(z[assoc[batch.src]], z[assoc[batch.neg_dst]])

        y_pred = torch.cat([pos, neg]).sigmoid().cpu()
        y_true = torch.cat([
            torch.ones(pos.size(0)),
            torch.zeros(neg.size(0)),
        ])

        aps.append(average_precision_score(y_true, y_pred))
        aucs.append(roc_auc_score(y_true, y_pred))
        mrrs.append(compute_mrr(pos.squeeze().cpu(), neg.squeeze().cpu()))

        memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        neighbor_loader.insert(batch.src, batch.dst)

    return (
        float(torch.tensor(aps).mean()),
        float(torch.tensor(aucs).mean()),
        float(np.mean(mrrs)),
    )

def run_experiment(name, train_td, val_data, test_data, data, epochs=40):
    loader      = TemporalDataLoader(train_td,  batch_size=200, neg_sampling_ratio=1.0)
    val_loader  = TemporalDataLoader(val_data,  batch_size=200, neg_sampling_ratio=1.0)
    test_loader = TemporalDataLoader(test_data, batch_size=200, neg_sampling_ratio=1.0)

    memory, gnn, link_pred, opt, assoc = create_model(
        data.num_nodes, data.msg.size(-1)
    )
    neigh = LastNeighborLoader(data.num_nodes, size=10, device=device)

    for ep in range(1, epochs + 1):
        loss = train_epoch(loader, memory, gnn, link_pred, opt, assoc, neigh, data)
        val_ap,  val_auc,  val_mrr  = eval_epoch(val_loader,  memory, gnn, link_pred, assoc, neigh, data)
        test_ap, test_auc, test_mrr = eval_epoch(test_loader, memory, gnn, link_pred, assoc, neigh, data)
        print(
            f"[{name}] Ep {ep:02d} | Loss {loss:.4f} "
            f"| Val AP {val_ap:.4f} "
            f"| Test AP {test_ap:.4f} AUC {test_auc:.4f} MRR {test_mrr:.4f}"
        )

    return test_ap, test_auc, test_mrr

# ---------------------------------------------------------------------------
# Compression Strategies
# ---------------------------------------------------------------------------

def random_compress(td, ratio=0.1, seed=42):
    torch.manual_seed(seed)
    k = int(ratio * td.num_events)
    idx = torch.randperm(td.num_events)[:k]
    if td.t.device != idx.device:
        idx = idx.to(td.t.device)
    idx = idx[td.t[idx].argsort()]

    return TemporalData(
        src=td.src[idx],
        dst=td.dst[idx],
        t=td.t[idx],
        msg=td.msg[idx],
    )

def synthetic_burst_compress(td, ratio=0.1, time_threshold=100.0, seed=42):
    print("--- Creating Synthetic Coreset (Burst Merging) ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": range(len(td.src)),
    }).sort_values(["src", "dst", "t"])

    time_diff = df["t"].diff().fillna(time_threshold + 1)
    src_diff  = df["src"].diff().fillna(1)
    dst_diff  = df["dst"].diff().fillna(1)
    is_new    = (src_diff != 0) | (dst_diff != 0) | (time_diff > time_threshold)
    df["burst_id"] = is_new.cumsum()

    num_bursts = df["burst_id"].max()
    print(f"   Found {num_bursts} bursts in {len(df)} events.")

    sorted_indices = torch.tensor(df["idx"].values,       dtype=torch.long)
    burst_ids      = torch.tensor(df["burst_id"].values,  dtype=torch.long) - 1

    msg_dim    = td.msg.size(1)
    sorted_msg = td.msg[sorted_indices].to(dev)

    meta_df  = df.groupby("burst_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()
    new_msg  = torch.zeros((len(meta_df), msg_dim),  device=dev, dtype=msg_dtype)
    counts   = torch.zeros((len(meta_df), 1),        device=dev, dtype=msg_dtype)

    new_msg.index_add_(0, burst_ids.to(dev), sorted_msg)
    counts.index_add_(0, burst_ids.to(dev), torch.ones_like(burst_ids, dtype=msg_dtype, device=dev).unsqueeze(1))
    new_msg = new_msg / counts

    new_src = torch.tensor(meta_df["src"].values, dtype=torch.long)
    new_dst = torch.tensor(meta_df["dst"].values, dtype=torch.long)
    new_t   = torch.tensor(meta_df["t"].values,   dtype=t_dtype)

    current_size = len(new_src)
    target_size  = int(td.num_events * ratio)

    if current_size > target_size:
        print(f"   Merged to {current_size}. Sampling down to {target_size}...")
        torch.manual_seed(seed)
        perm       = torch.randperm(current_size)[:target_size]
        sort_order = new_t[perm].argsort()
        final_idx  = perm[sort_order]
        new_src, new_dst, new_t, new_msg = (
            new_src[final_idx], new_dst[final_idx],
            new_t[final_idx],   new_msg[final_idx],
        )
    else:
        print(f"   Merging reached target ({current_size} events).")

    return TemporalData(src=new_src, dst=new_dst, t=new_t, msg=new_msg.cpu())

def _train_msg_model(td, num_nodes, epochs=3, batch_size=512):
    model   = MessageDstPredictor(num_nodes, td.msg.size(-1)).to(device)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    td_dev = td.to(device)
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(td_dev.num_events, device=device)
        for i in range(0, td_dev.num_events, batch_size):
            idx     = perm[i : i + batch_size]
            targets = td_dev.dst[idx].clamp(0, num_nodes - 1)
            opt.zero_grad()
            loss = loss_fn(model(td_dev.src[idx], td_dev.msg[idx]), targets)
            loss.backward()
            opt.step()

    model.eval()
    return model

def learned_compress(td, num_nodes, ratio=0.1):
    td = td.to(device)
    msg_model = _train_msg_model(td, num_nodes)

    # Message surprisal
    msg_scores = torch.zeros(td.num_events, device=device)
    with torch.no_grad():
        for i in range(td.num_events):
            logits  = msg_model(td.src[i : i + 1], td.msg[i : i + 1])
            dst_idx = td.dst[i].clamp(0, num_nodes - 1)
            msg_scores[i] = -torch.log_softmax(logits, dim=-1)[0, dst_idx]
    msg_scores = (msg_scores - msg_scores.min()) / (msg_scores.max() - msg_scores.min() + 1e-8)

    # Temporal surprisal
    temporal_scores = torch.zeros(td.num_events, device=device)
    last_time: dict = {}
    for i in range(td.num_events):
        key = (td.src[i].item(), td.dst[i].item())
        t   = td.t[i].item()
        dt  = t - last_time[key] if key in last_time else 1.0
        temporal_scores[i] = torch.log1p(torch.tensor(dt, device=device))
        last_time[key] = t
    temporal_scores = (temporal_scores - temporal_scores.min()) / (temporal_scores.max() - temporal_scores.min() + 1e-8)

    scores = 0.7 * msg_scores + 0.3 * temporal_scores

    # Per-source quota selection
    src_to_events: dict = defaultdict(list)
    for i in range(td.num_events):
        src_to_events[td.src[i].item()].append(i)

    selected = []
    for _, evs in src_to_events.items():
        evs_t = torch.tensor(evs, device=device)
        k     = max(1, int(ratio * len(evs_t)))
        top   = evs_t[torch.argsort(scores[evs_t], descending=True)[:k]]
        selected.append(top)

    idx    = torch.cat(selected).unique()
    target = int(ratio * td.num_events)
    if idx.numel() > target:
        idx = idx[torch.argsort(scores[idx], descending=True)[:target]]
    idx = idx.sort().values
    print(f"[Learned] Selected {idx.numel()} events")

    return TemporalData(
        src=td.src[idx], dst=td.dst[idx],
        t=td.t[idx],     msg=td.msg[idx],
    )

def motif_based_compress(td, window_size=1000.0):
    print(f"--- Mining Temporal Motifs (window={window_size}s) ---")
    dev        = td.msg.device
    num_events = len(td.src)
    t_numpy    = td.t.cpu().numpy()
    buckets    = (t_numpy // window_size).astype(int)

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   t_numpy,
        "bucket": buckets,
        "global_idx": range(num_events),
    })

    df["motif_id"]    = -1
    current_motif_max = 0

    for _, group in df.groupby("bucket"):
        u, v = group["src"].values, group["dst"].values
        unique_nodes, inverse = np.unique(np.concatenate([u, v]), return_inverse=True)
        local_u, local_v = inverse[: len(u)], inverse[len(u) :]

        adj = sp.coo_matrix(
            (np.ones(len(local_u)), (local_u, local_v)),
            shape=(len(unique_nodes), len(unique_nodes)),
        )
        n_comp, labels = sp.csgraph.connected_components(adj, connection="weak", directed=False)
        df.loc[group.index, "motif_id"] = labels[local_u] + current_motif_max
        current_motif_max += n_comp

    print(f"   Found {current_motif_max} temporal motifs in {num_events} events.")

    burst_ids   = torch.tensor(df["motif_id"].values, device=dev, dtype=torch.long)
    sorted_idx  = torch.tensor(df["global_idx"].values, device=dev)
    sorted_msg  = td.msg[sorted_idx]

    new_msg = torch.zeros((current_motif_max, td.msg.shape[1]), device=dev, dtype=td.msg.dtype)
    counts  = torch.zeros((current_motif_max, 1),               device=dev, dtype=td.msg.dtype)
    new_msg.index_add_(0, burst_ids, sorted_msg)
    counts.index_add_(0, burst_ids, torch.ones_like(burst_ids, dtype=td.msg.dtype).unsqueeze(1))
    new_msg = new_msg / counts

    meta_df = df.groupby("motif_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()

    return TemporalData(
        src=torch.tensor(meta_df["src"].values, dtype=torch.long),
        dst=torch.tensor(meta_df["dst"].values, dtype=torch.long),
        t=torch.tensor(meta_df["t"].values,     dtype=td.t.dtype),
        msg=new_msg.cpu(),
    )

def _score_motif_importance(motif_td, num_nodes, msg_dim):
    print("--- Scoring Motifs (Contrastive Difficulty) ---")
    motif_td = motif_td.to(device)

    mem_dim = emb_dim = 64
    memory = TGNMemory(
        num_nodes, msg_dim, mem_dim, mem_dim,
        message_module=IdentityMessage(msg_dim, mem_dim, mem_dim),
        aggregator_module=LastAggregator(),
    ).to(device)
    memory.reset_state()

    gnn       = GraphAttentionEmbedding(mem_dim, emb_dim, msg_dim, memory.time_enc).to(device)
    link_pred = LinkPredictor(emb_dim).to(device)
    optimizer = torch.optim.Adam(
        list(memory.parameters()) + list(gnn.parameters()) + list(link_pred.parameters()),
        lr=1e-3,
    )

    neighbor_loader = LastNeighborLoader(num_nodes, size=10, device=device)
    neighbor_loader.reset_state()
    criterion = nn.BCEWithLogitsLoss(reduction="none")

    losses     = torch.zeros(motif_td.num_events, device=device)
    batch_size = 512

    memory.train(); gnn.train(); link_pred.train()

    for i in range(0, motif_td.num_events, batch_size):
        end       = min(i + batch_size, motif_td.num_events)
        batch_idx = torch.arange(i, end, device=device)

        src, dst, t, msg = (
            motif_td.src[batch_idx], motif_td.dst[batch_idx],
            motif_td.t[batch_idx],   motif_td.msg[batch_idx],
        )
        neg_dst = torch.randint(0, num_nodes, (batch_idx.size(0),), device=device)

        n_id          = torch.cat([src, dst, neg_dst]).unique()
        n_id, e_idx, e_id = neighbor_loader(n_id)
        assoc         = torch.empty(num_nodes, dtype=torch.long, device=device)
        assoc[n_id]   = torch.arange(n_id.size(0), device=device)

        z, last   = memory(n_id)
        z         = gnn(z, last, e_idx, motif_td.t[e_id], motif_td.msg[e_id])

        pos_out   = link_pred(z[assoc[src]], z[assoc[dst]])
        neg_out   = link_pred(z[assoc[src]], z[assoc[neg_dst]])
        batch_loss = (criterion(pos_out, torch.ones_like(pos_out)) +
                      criterion(neg_out, torch.zeros_like(neg_out))).squeeze()

        losses[batch_idx] = batch_loss.detach()

        batch_loss.mean().backward()
        optimizer.step(); optimizer.zero_grad()

        memory.detach()
        memory.update_state(src, dst, t, msg)
        neighbor_loader.insert(src, dst)

    return losses

def subgraph_motif_compress(td, num_nodes, msg_dim, ratio=0.1, window_size=2000.0):
    motif_td    = motif_based_compress(td, window_size=window_size)
    target_size = int(td.num_events * ratio)

    if motif_td.num_events <= target_size:
        print(f"   Motifs ({motif_td.num_events}) already within budget.")
        return motif_td

    losses    = _score_motif_importance(motif_td, num_nodes, msg_dim)
    norm_loss = (losses - losses.min()) / (losses.max() - losses.min() + 1e-8)
    weights   = 0.7 * norm_loss + 0.3

    print(f"--- Sampling {target_size} motifs from {motif_td.num_events} ---")
    torch.manual_seed(42)
    sampled_idx = torch.multinomial(weights, target_size, replacement=False)
    sorted_idx  = sampled_idx[motif_td.t[sampled_idx].argsort()]

    return TemporalData(
        src=motif_td.src[sorted_idx],
        dst=motif_td.dst[sorted_idx],
        t=motif_td.t[sorted_idx],
        msg=motif_td.msg[sorted_idx],
    )
# ---------------------------------------------------------------------------
# Evaluations
# ---------------------------------------------------------------------------
def compute_mrr(pos_scores: torch.Tensor, neg_scores: torch.Tensor) -> float:
    y_pred_pos = pos_scores.numpy().reshape(-1, 1)
    y_pred_neg = neg_scores.numpy().reshape(-1, 1)

    optimistic_rank = (y_pred_neg > y_pred_pos).sum(axis=1)
    pessimistic_rank = (y_pred_neg >= y_pred_pos).sum(axis=1)
    ranking_list = 0.5 * (optimistic_rank + pessimistic_rank) + 1
    mrr_list = 1.0 / ranking_list.astype(np.float32)
    return float(mrr_list.mean())



# ---------------------------------------------------------------------------
# Results Printing
# ---------------------------------------------------------------------------

def print_results(results: dict):
    print(f"{'METHOD':<16} | {'AP':<8} | {'AUC':<8} | {'MRR':<8}")
    for k, (ap, auc, mrr) in results.items():
        print(f"{k:<16} | {ap:.4f}   | {auc:.4f}   | {mrr:.4f}")
"""

In [7]:
with open("utils.py", "w") as f:
    f.write(code)

In [12]:
# --- Imports ---
import os.path as osp
import torch
from torch_geometric.datasets import JODIEDataset

from utils import (
    device,
    run_experiment,
    random_compress,
    learned_compress,
    synthetic_burst_compress,
    subgraph_motif_compress,
    print_results,
)

print("Using device:", device)

Using device: cuda


In [13]:
def recency_weighted_burst_compress(td, ratio=0.1, time_threshold=1000.0, 
                                     decay_from_end=True, seed=42):
    # decay_from_end=True  -> last event in burst gets weight 1.0, earlier decay back
    # decay_from_end=False -> first event gets weight 1.0, later events decay forward
    print(f"--- Recency-Weighted Burst Compress (decay_from_end={decay_from_end}) ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": range(len(td.src)),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    time_diff = df["t"].diff().fillna(time_threshold + 1)
    src_diff  = df["src"].diff().fillna(1)
    dst_diff  = df["dst"].diff().fillna(1)
    is_new    = (src_diff != 0) | (dst_diff != 0) | (time_diff > time_threshold)
    df["burst_id"] = is_new.cumsum() - 1  # 0-indexed

    num_bursts = df["burst_id"].max() + 1
    print(f"   Found {num_bursts} bursts in {len(df)} events.")

    msg_dim = td.msg.size(1)
    sorted_indices = torch.tensor(df["idx"].values, dtype=torch.long)
    burst_ids      = torch.tensor(df["burst_id"].values, dtype=torch.long)
    sorted_msg     = td.msg[sorted_indices].to(dev)
    sorted_t       = torch.tensor(df["t"].values, dtype=torch.float32).to(dev)

    new_msg  = torch.zeros((num_bursts, msg_dim), device=dev, dtype=msg_dtype)
    weight_sum = torch.zeros((num_bursts, 1), device=dev, dtype=torch.float32)

    # compute per-event decay weights using delta-t within each burst
    burst_ids_dev = burst_ids.to(dev)

    # anchor timestamp per burst: last t if decay_from_end, first t if not
    anchor_t = torch.zeros(num_bursts, device=dev, dtype=torch.float32)
    if decay_from_end:
        anchor_t.scatter_reduce_(0, burst_ids_dev, sorted_t, reduce="amax", include_self=True)
    else:
        anchor_t.scatter_reduce_(0, burst_ids_dev, sorted_t, reduce="amin", include_self=True)

    # delta from anchor, always positive
    delta = (anchor_t[burst_ids_dev] - sorted_t).abs()  # shape [num_events]

    # normalize delta per burst to get stable decay input
    burst_max_delta = torch.zeros(num_bursts, device=dev, dtype=torch.float32)
    burst_max_delta.scatter_reduce_(0, burst_ids_dev, delta, reduce="amax", include_self=True)
    burst_max_delta = burst_max_delta.clamp(min=1.0)  # avoid div by zero in single-event bursts

    norm_delta = delta / burst_max_delta[burst_ids_dev]  # in [0, 1]
    weights    = torch.exp(-3.0 * norm_delta).unsqueeze(1)  # λ=3, sharper near anchor

    weighted_msg = sorted_msg.float() * weights

    new_msg_f   = torch.zeros((num_bursts, msg_dim), device=dev, dtype=torch.float32)
    new_msg_f.index_add_(0, burst_ids_dev, weighted_msg)
    weight_sum.index_add_(0, burst_ids_dev, weights)

    new_msg = (new_msg_f / weight_sum.clamp(min=1e-8)).to(msg_dtype)

    # representative timestamp per burst
    meta_df = df.groupby("burst_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()
    new_src = torch.tensor(meta_df["src"].values, dtype=torch.long)
    new_dst = torch.tensor(meta_df["dst"].values, dtype=torch.long)
    new_t   = torch.tensor(meta_df["t"].values,   dtype=t_dtype)

    # sample down to budget if needed
    current_size = len(new_src)
    target_size  = int(td.num_events * ratio)

    if current_size > target_size:
        print(f"   Merged to {current_size}. Sampling down to {target_size}...")
        torch.manual_seed(seed)
        perm       = torch.randperm(current_size)[:target_size]
        sort_order = new_t[perm].argsort()
        final_idx  = perm[sort_order]
        new_src, new_dst, new_t, new_msg = (
            new_src[final_idx], new_dst[final_idx],
            new_t[final_idx],   new_msg[final_idx],
        )
    else:
        print(f"   Merging reached target ({current_size} events).")

    return TemporalData(src=new_src, dst=new_dst, t=new_t, msg=new_msg.cpu())

def delta_t_weighted_burst_compress(td, ratio=0.1, time_threshold=1000.0, seed=42):
    print("--- Delta-t Weighted Burst Compress ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": range(len(td.src)),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    # delta-t between consecutive events per (src, dst) pair
    df["dt"] = df.groupby(["src", "dst"])["t"].diff().fillna(0.0)

    time_diff = df["t"].diff().fillna(time_threshold + 1)
    src_diff  = df["src"].diff().fillna(1)
    dst_diff  = df["dst"].diff().fillna(1)
    is_new    = (src_diff != 0) | (dst_diff != 0) | (time_diff > time_threshold)
    df["burst_id"] = is_new.cumsum() - 1

    num_bursts = df["burst_id"].max() + 1
    print(f"   Found {num_bursts} bursts in {len(df)} events.")

    msg_dim        = td.msg.size(1)
    sorted_indices = torch.tensor(df["idx"].values,      dtype=torch.long)
    burst_ids      = torch.tensor(df["burst_id"].values, dtype=torch.long).to(dev)
    sorted_msg     = td.msg[sorted_indices].to(dev)

    # weight = log1p(dt) so the first event (dt=0) gets weight log1p(0)=0... 
    # bump it to 1.0 so no event is fully ignored
    raw_dt  = torch.tensor(df["dt"].values, dtype=torch.float32).to(dev)
    weights = torch.log1p(raw_dt).clamp(min=1.0).unsqueeze(1)  # [num_events, 1]

    weighted_msg = sorted_msg.float() * weights
    new_msg_f    = torch.zeros((num_bursts, msg_dim), device=dev, dtype=torch.float32)
    weight_sum   = torch.zeros((num_bursts, 1),       device=dev, dtype=torch.float32)

    new_msg_f.index_add_(0, burst_ids, weighted_msg)
    weight_sum.index_add_(0, burst_ids, weights)

    new_msg = (new_msg_f / weight_sum.clamp(min=1e-8)).to(msg_dtype)

    meta_df = df.groupby("burst_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()
    new_src = torch.tensor(meta_df["src"].values, dtype=torch.long)
    new_dst = torch.tensor(meta_df["dst"].values, dtype=torch.long)
    new_t   = torch.tensor(meta_df["t"].values,   dtype=t_dtype)

    current_size = len(new_src)
    target_size  = int(td.num_events * ratio)

    if current_size > target_size:
        print(f"   Merged to {current_size}. Sampling down to {target_size}...")
        torch.manual_seed(seed)
        perm       = torch.randperm(current_size)[:target_size]
        sort_order = new_t[perm].argsort()
        final_idx  = perm[sort_order]
        new_src, new_dst, new_t, new_msg = (
            new_src[final_idx], new_dst[final_idx],
            new_t[final_idx],   new_msg[final_idx],
        )
    else:
        print(f"   Merging reached target ({current_size} events).")

    return TemporalData(src=new_src, dst=new_dst, t=new_t, msg=new_msg.cpu())

In [14]:
def compute_adaptive_windows(td, stat="mean"):
    """
    Compute per-edge adaptive time_threshold = max(src_window, dst_window)
    where window = mean or median of inter-arrival δt for each node.
    Fallback for single-event nodes = global mean/median of all δt.
    
    Returns: torch.Tensor of shape [num_events] with per-event thresholds
             (aligned to df sorted by src, dst, t)
    """
    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    # δt per node (treating each node as src across all its edges)
    # Stack src and dst together to get all node appearances
    src_df = df[["src", "t"]].rename(columns={"src": "node"})
    dst_df = df[["dst", "t"]].rename(columns={"dst": "node"})
    node_df = pd.concat([src_df, dst_df]).sort_values(["node", "t"]).reset_index(drop=True)
    node_df["dt"] = node_df.groupby("node")["t"].diff()

    if stat == "mean":
        node_window = node_df.groupby("node")["dt"].mean()
        global_fallback = node_df["dt"].mean()
    elif stat == "median":
        node_window = node_df.groupby("node")["dt"].median()
        global_fallback = node_df["dt"].median()
    else:
        raise ValueError(f"stat must be 'mean' or 'median', got {stat}")

    # fill NaN (single-event nodes) with global fallback
    node_window = node_window.fillna(global_fallback)
    node_window_dict = node_window.to_dict()

    # per-edge threshold = max(src_window, dst_window)
    src_w = df["src"].map(node_window_dict).fillna(global_fallback).values
    dst_w = df["dst"].map(node_window_dict).fillna(global_fallback).values
    edge_threshold = np.maximum(src_w, dst_w)

    return edge_threshold  # shape [num_events], aligned to sorted df


def adaptive_recency_burst_compress(td, ratio=0.1, stat="mean", decay_from_end=True, seed=42):
    print(f"--- Adaptive Recency Burst Compress (stat={stat}, decay_from_end={decay_from_end}) ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": range(len(td.src)),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    # adaptive per-edge thresholds
    edge_threshold = compute_adaptive_windows(td, stat=stat)  # [num_events], sorted order

    time_diff = df["t"].diff().fillna(edge_threshold[0] + 1)
    src_diff  = df["src"].diff().fillna(1)
    dst_diff  = df["dst"].diff().fillna(1)

    # burst break when δt > this edge's adaptive threshold
    time_exceeds = pd.Series(
        np.abs(df["t"].diff().fillna(np.inf).values) > edge_threshold
    )
    is_new = (src_diff != 0) | (dst_diff != 0) | time_exceeds
    df["burst_id"] = is_new.cumsum() - 1

    num_bursts = df["burst_id"].max() + 1
    print(f"   Found {num_bursts} bursts in {len(df)} events (stat={stat}).")

    msg_dim        = td.msg.size(1)
    sorted_indices = torch.tensor(df["idx"].values,      dtype=torch.long)
    burst_ids_dev  = torch.tensor(df["burst_id"].values, dtype=torch.long).to(dev)
    sorted_msg     = td.msg[sorted_indices].to(dev)
    sorted_t       = torch.tensor(df["t"].values, dtype=torch.float32).to(dev)

    # recency weights (same as recency_weighted_burst_compress)
    anchor_t = torch.zeros(num_bursts, device=dev, dtype=torch.float32)
    if decay_from_end:
        anchor_t.scatter_reduce_(0, burst_ids_dev, sorted_t, reduce="amax", include_self=True)
    else:
        anchor_t.scatter_reduce_(0, burst_ids_dev, sorted_t, reduce="amin", include_self=True)

    delta          = (anchor_t[burst_ids_dev] - sorted_t).abs()
    burst_max_delta = torch.zeros(num_bursts, device=dev, dtype=torch.float32)
    burst_max_delta.scatter_reduce_(0, burst_ids_dev, delta, reduce="amax", include_self=True)
    burst_max_delta = burst_max_delta.clamp(min=1.0)

    norm_delta = delta / burst_max_delta[burst_ids_dev]
    weights    = torch.exp(-3.0 * norm_delta).unsqueeze(1)

    weighted_msg = sorted_msg.float() * weights
    new_msg_f    = torch.zeros((num_bursts, msg_dim), device=dev, dtype=torch.float32)
    weight_sum   = torch.zeros((num_bursts, 1),       device=dev, dtype=torch.float32)
    new_msg_f.index_add_(0, burst_ids_dev, weighted_msg)
    weight_sum.index_add_(0, burst_ids_dev, weights)
    new_msg = (new_msg_f / weight_sum.clamp(min=1e-8)).to(msg_dtype)

    meta_df = df.groupby("burst_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()
    new_src = torch.tensor(meta_df["src"].values, dtype=torch.long)
    new_dst = torch.tensor(meta_df["dst"].values, dtype=torch.long)
    new_t   = torch.tensor(meta_df["t"].values,   dtype=t_dtype)

    current_size = len(new_src)
    target_size  = int(td.num_events * ratio)

    if current_size > target_size:
        print(f"   Merged to {current_size}. Sampling down to {target_size}...")
        torch.manual_seed(seed)
        perm       = torch.randperm(current_size)[:target_size]
        sort_order = new_t[perm].argsort()
        final_idx  = perm[sort_order]
        new_src, new_dst, new_t, new_msg = (
            new_src[final_idx], new_dst[final_idx],
            new_t[final_idx],   new_msg[final_idx],
        )
    else:
        print(f"   Merging reached target ({current_size} events).")

    return TemporalData(src=new_src, dst=new_dst, t=new_t, msg=new_msg.cpu())


def adaptive_delta_t_burst_compress(td, ratio=0.1, stat="mean", seed=42):
    print(f"--- Adaptive Delta-t Burst Compress (stat={stat}) ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": range(len(td.src)),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    edge_threshold = compute_adaptive_windows(td, stat=stat)

    df["dt"] = df.groupby(["src", "dst"])["t"].diff().fillna(0.0)

    time_exceeds = pd.Series(
        np.abs(df["t"].diff().fillna(np.inf).values) > edge_threshold
    )
    src_diff = df["src"].diff().fillna(1)
    dst_diff = df["dst"].diff().fillna(1)
    is_new   = (src_diff != 0) | (dst_diff != 0) | time_exceeds
    df["burst_id"] = is_new.cumsum() - 1

    num_bursts = df["burst_id"].max() + 1
    print(f"   Found {num_bursts} bursts in {len(df)} events (stat={stat}).")

    msg_dim        = td.msg.size(1)
    sorted_indices = torch.tensor(df["idx"].values,      dtype=torch.long)
    burst_ids      = torch.tensor(df["burst_id"].values, dtype=torch.long).to(dev)
    sorted_msg     = td.msg[sorted_indices].to(dev)

    raw_dt  = torch.tensor(df["dt"].values, dtype=torch.float32).to(dev)
    weights = torch.log1p(raw_dt).clamp(min=1.0).unsqueeze(1)

    weighted_msg = sorted_msg.float() * weights
    new_msg_f    = torch.zeros((num_bursts, msg_dim), device=dev, dtype=torch.float32)
    weight_sum   = torch.zeros((num_bursts, 1),       device=dev, dtype=torch.float32)
    new_msg_f.index_add_(0, burst_ids, weighted_msg)
    weight_sum.index_add_(0, burst_ids, weights)
    new_msg = (new_msg_f / weight_sum.clamp(min=1e-8)).to(msg_dtype)

    meta_df = df.groupby("burst_id").agg({"src": "first", "dst": "first", "t": "last"}).reset_index()
    new_src = torch.tensor(meta_df["src"].values, dtype=torch.long)
    new_dst = torch.tensor(meta_df["dst"].values, dtype=torch.long)
    new_t   = torch.tensor(meta_df["t"].values,   dtype=t_dtype)

    current_size = len(new_src)
    target_size  = int(td.num_events * ratio)

    if current_size > target_size:
        print(f"   Merged to {current_size}. Sampling down to {target_size}...")
        torch.manual_seed(seed)
        perm       = torch.randperm(current_size)[:target_size]
        sort_order = new_t[perm].argsort()
        final_idx  = perm[sort_order]
        new_src, new_dst, new_t, new_msg = (
            new_src[final_idx], new_dst[final_idx],
            new_t[final_idx],   new_msg[final_idx],
        )
    else:
        print(f"   Merging reached target ({current_size} events).")

    return TemporalData(src=new_src, dst=new_dst, t=new_t, msg=new_msg.cpu())

In [15]:
import pandas as pd

def temporal_to_df(data):
    df = pd.DataFrame({
        "src": data.src.cpu().numpy(),
        "dst": data.dst.cpu().numpy(),
        "t": data.t.cpu().numpy(),
    })

    # Optional: include edge features if present
    if hasattr(data, "msg") and data.msg is not None:
        msg = data.msg.cpu().numpy()
        for i in range(msg.shape[1]):
            df[f"msg_{i}"] = msg[:, i]

    return df

In [19]:
import os
os.makedirs("compressed_csv", exist_ok=True)

# Random
rand_data = random_compress(train_data, ratio=0.1)
temporal_to_df(rand_data).to_csv("compressed_csv/random_10.csv", index=False)

# Recency end
rec_end = recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=True)
temporal_to_df(rec_end).to_csv("compressed_csv/recency_end_10.csv", index=False)

# Recency start
rec_start = recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=False)
temporal_to_df(rec_start).to_csv("compressed_csv/recency_start_10.csv", index=False)

# Delta-t
delta_t = delta_t_weighted_burst_compress(train_data, ratio=0.1)
temporal_to_df(delta_t).to_csv("compressed_csv/delta_t_10.csv", index=False)

--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...
--- Recency-Weighted Burst Compress (decay_from_end=False) ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...
--- Delta-t Weighted Burst Compress ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...


In [22]:
# --- Imports ---
import torch
import os
from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset
from torch_geometric.data import TemporalData

from utils import (
    device,
    run_experiment,
    random_compress,
    print_results,
)

print("Using device:", device)

DATASETS = ["tgbl-uci"]
all_results = {}

# Create directory
os.makedirs("compressed", exist_ok=True)

for dataset_name in DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
    data = dataset.get_TemporalData()

    train_mask = dataset.train_mask
    val_mask   = dataset.val_mask
    test_mask  = dataset.test_mask

    train_data = data[train_mask]
    val_data   = data[val_mask]
    test_data  = data[test_mask]

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # =======================
    # 1. FULL DATA
    # =======================
    print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # =======================
    # 2. RANDOM 10%
    # =======================
    rand_path = "compressed/random_10.pt"

    if os.path.exists(rand_path):
        print("Loading RANDOM 10% from cache...")
        rand_data = torch.load(rand_path).to(device)
    else:
        print("Computing RANDOM 10%...")
        rand_data = random_compress(train_data, ratio=0.1)
        torch.save(rand_data.cpu(), rand_path)

    # results["random_10"] = run_experiment(
    #     "RAND10", rand_data, val_data, test_data, data
    # )

    # =======================
    # 3. RECENCY END
    # =======================
    rec_end_path = "compressed/recency_end_10.pt"

    if os.path.exists(rec_end_path):
        print("Loading RECENCY END from cache...")
        rec_end = torch.load(rec_end_path).to(device)
    else:
        print("Computing RECENCY END...")
        rec_end = recency_weighted_burst_compress(
            train_data, ratio=0.1, decay_from_end=True
        )
        torch.save(rec_end.cpu(), rec_end_path)

    # results["recency_end_10"] = run_experiment(
    #     "REC_END", rec_end, val_data, test_data, data
    # )

    # =======================
    # 4. RECENCY START
    # =======================
    rec_start_path = "compressed/recency_start_10.pt"

    if os.path.exists(rec_start_path):
        print("Loading RECENCY START from cache...")
        rec_start = torch.load(rec_start_path).to(device)
    else:
        print("Computing RECENCY START...")
        rec_start = recency_weighted_burst_compress(
            train_data, ratio=0.1, decay_from_end=False
        )
        torch.save(rec_start.cpu(), rec_start_path)

    # results["recency_start_10"] = run_experiment(
    #     "REC_START", rec_start, val_data, test_data, data
    # )

    # =======================
    # 5. DELTA-T
    # =======================
    delta_path = "compressed/delta_t_10.pt"

    if os.path.exists(delta_path):
        print("Loading DELTA-T from cache...")
        delta_t = torch.load(delta_path).to(device)
    else:
        print("Computing DELTA-T...")
        delta_t = delta_t_weighted_burst_compress(train_data, ratio=0.1)
        torch.save(delta_t.cpu(), delta_path)

    # results["delta_t_10"] = run_experiment(
    #     "DELTA_T", delta_t, val_data, test_data, data
    # )

    # all_results[dataset_name] = results


# =======================
# PRINT RESULTS
# =======================
print_results(all_results)

Using device: cuda

RUNNING DATASET: TGBL-UCI
Total events: 59835 | Train: 41884 | Val: 8975 | Test: 8976

=== FULL DATA ===
Computing RANDOM 10%...
Computing RECENCY END...
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...
Computing RECENCY START...
--- Recency-Weighted Burst Compress (decay_from_end=False) ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...
Computing DELTA-T...
--- Delta-t Weighted Burst Compress ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...
METHOD           | AP       | AUC      | MRR     


In [24]:
# --- Imports ---
import os
import os.path as osp
import torch
from torch_geometric.datasets import JODIEDataset

from utils import (
    device,
    run_experiment,
    random_compress,
    print_results,
)

print("Using device:", device)

# --------------------------------------------------
# Datasets to run
# --------------------------------------------------
DATASETS = ["wikipedia", "reddit", "mooc"]

all_results = {}

# Create cache directory
os.makedirs("compressed_jodie", exist_ok=True)

# --------------------------------------------------
# Loop over datasets
# --------------------------------------------------
for dataset_name in DATASETS:

    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    # --- Split ---
    train_data, val_data, test_data = data.train_val_test_split(
        val_ratio=0.15,
        test_ratio=0.15
    )

    # --- Move to device ---
    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # ==================================================
    # 1. FULL DATA
    # ==================================================
    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment(
    #     "FULL", train_data, val_data, test_data, data
    # )

    # ==================================================
    # 2. RANDOM 10% (with caching)
    # ==================================================
    rand_path = f"compressed_jodie/{dataset_name}_random_10.pt"

    if os.path.exists(rand_path):
        print("Loading RANDOM 10% from cache...")
        rand_data = torch.load(rand_path).to(device)
    else:
        print("Computing RANDOM 10%...")
        rand_data = random_compress(train_data, ratio=0.1)
        torch.save(rand_data.cpu(), rand_path)

    results["random_10"] = run_experiment(
        "RAND10",
        rand_data,
        val_data,
        test_data,
        data,
    )

    # Save per-dataset results
    all_results[dataset_name] = results


# --------------------------------------------------
# Print final summary
# --------------------------------------------------
print_results(all_results)

Using device: cuda

RUNNING DATASET: WIKIPEDIA
Total events: 157474 | Train: 110232 | Val: 23621 | Test: 23621
Computing RANDOM 10%...


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[RAND10] Ep 01 | Loss 1.3098 | Val AP 0.7871 | Test AP 0.7583 AUC 0.7571 MRR 0.9000
[RAND10] Ep 02 | Loss 1.2081 | Val AP 0.8470 | Test AP 0.7892 AUC 0.8010 MRR 0.9195


KeyboardInterrupt: 

In [14]:
# --- Imports ---
import torch
from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset
from utils import (
    device,
    run_experiment,
    random_compress,
    learned_compress,
    synthetic_burst_compress,
    subgraph_motif_compress,
    print_results,
)
print("Using device:", device)

DATASETS = ["tgbl-uci"]
all_results = {}

for dataset_name in DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
    data = dataset.get_TemporalData()

    train_mask = dataset.train_mask
    val_mask   = dataset.val_mask
    test_mask  = dataset.test_mask

    train_data = data[train_mask]
    val_data   = data[val_mask]
    test_data  = data[test_mask]

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # # 1. Full data baseline
    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # # 2. Random 10%
    # print("\n=== RANDOM 10% ===")
    # results["random_10"] = run_experiment(
    #     "RAND10", random_compress(train_data, ratio=0.1), val_data, test_data, data
    # )

    # # 3. Recency burst (decay from end) 10%
    # print("\n=== RECENCY BURST (decay from end) 10% ===")
    # results["recency_end_10"] = run_experiment(
    #     "REC_END",
    #     recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=True),
    #     val_data, test_data, data
    # )

    # # 4. Recency burst (decay from start) 10%
    # print("\n=== RECENCY BURST (decay from start) 10% ===")
    # results["recency_start_10"] = run_experiment(
    #     "REC_START",
    #     recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=False),
    #     val_data, test_data, data
    # )

    # # 5. Delta-t burst 10%
    # print("\n=== DELTA-T BURST 10% ===")
    # results["delta_t_10"] = run_experiment(
    #     "DELTA_T",
    #     delta_t_weighted_burst_compress(train_data, ratio=0.1),
    #     val_data, test_data, data
    # )

    # all_results[dataset_name] = results

print_results(all_results)

Using device: cuda

RUNNING DATASET: TGBL-UCI
Total events: 59835 | Train: 41884 | Val: 8975 | Test: 8976

=== FULL DATA ===


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[FULL] Ep 01 | Loss 20.6462 | Val AP 0.5641 | Test AP 0.5656 AUC 0.6053 MRR 0.9076
[FULL] Ep 02 | Loss 8.6165 | Val AP 0.4926 | Test AP 0.4929 AUC 0.4852 MRR 0.8486
[FULL] Ep 03 | Loss 8.8539 | Val AP 0.6601 | Test AP 0.6838 AUC 0.7406 MRR 0.9123
[FULL] Ep 04 | Loss 5.9734 | Val AP 0.6946 | Test AP 0.7671 AUC 0.8097 MRR 0.9112
[FULL] Ep 05 | Loss 4.4321 | Val AP 0.6679 | Test AP 0.6509 AUC 0.7206 MRR 0.9327
[FULL] Ep 06 | Loss 3.9814 | Val AP 0.7001 | Test AP 0.8029 AUC 0.8182 MRR 0.9069
[FULL] Ep 07 | Loss 4.4698 | Val AP 0.6223 | Test AP 0.6190 AUC 0.6841 MRR 0.9321
[FULL] Ep 08 | Loss 4.9126 | Val AP 0.5122 | Test AP 0.5030 AUC 0.5046 MRR 0.9108
[FULL] Ep 09 | Loss 3.6638 | Val AP 0.7223 | Test AP 0.8238 AUC 0.8407 MRR 0.9281
[FULL] Ep 10 | Loss 2.7905 | Val AP 0.5104 | Test AP 0.5411 AUC 0.6147 MRR 0.8129
[FULL] Ep 11 | Loss 2.3482 | Val AP 0.7257 | Test AP 0.8338 AUC 0.8627 MRR 0.9406
[FULL] Ep 12 | Loss 2.3036 | Val AP 0.7416 | Test AP 0.8502 AUC 0.8625 MRR 0.9348
[FULL] Ep 13 | 

/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[RAND10] Ep 01 | Loss 2.1964 | Val AP 0.6846 | Test AP 0.7091 AUC 0.7740 MRR 0.9294
[RAND10] Ep 02 | Loss 1.9579 | Val AP 0.6859 | Test AP 0.7443 AUC 0.7918 MRR 0.9041
[RAND10] Ep 03 | Loss 2.0038 | Val AP 0.5023 | Test AP 0.4500 AUC 0.4501 MRR 0.7141
[RAND10] Ep 04 | Loss 2.3042 | Val AP 0.7162 | Test AP 0.7283 AUC 0.7876 MRR 0.9266
[RAND10] Ep 05 | Loss 1.7251 | Val AP 0.6767 | Test AP 0.7162 AUC 0.7380 MRR 0.8549
[RAND10] Ep 06 | Loss 1.8499 | Val AP 0.7143 | Test AP 0.8149 AUC 0.8382 MRR 0.9284
[RAND10] Ep 07 | Loss 1.5446 | Val AP 0.7128 | Test AP 0.8219 AUC 0.8117 MRR 0.8944
[RAND10] Ep 08 | Loss 1.4227 | Val AP 0.6231 | Test AP 0.6275 AUC 0.6326 MRR 0.8562
[RAND10] Ep 09 | Loss 1.6860 | Val AP 0.7375 | Test AP 0.8251 AUC 0.8442 MRR 0.9248
[RAND10] Ep 10 | Loss 1.6708 | Val AP 0.7133 | Test AP 0.7461 AUC 0.7935 MRR 0.8923
[RAND10] Ep 11 | Loss 1.5580 | Val AP 0.7512 | Test AP 0.8299 AUC 0.8688 MRR 0.9418
[RAND10] Ep 12 | Loss 1.7891 | Val AP 0.6200 | Test AP 0.5973 AUC 0.6007 MRR

NameError: name 'recency_weighted_burst_compress' is not defined

In [18]:
from torch_geometric.data import TemporalData

In [19]:
# --- Imports ---
import torch
from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset
from utils import (
    device,
    run_experiment,
    random_compress,
    learned_compress,
    synthetic_burst_compress,
    subgraph_motif_compress,
    print_results,
)
print("Using device:", device)

DATASETS = ["tgbl-uci"]
all_results = {}

for dataset_name in DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
    data = dataset.get_TemporalData()

    train_mask = dataset.train_mask
    val_mask   = dataset.val_mask
    test_mask  = dataset.test_mask

    train_data = data[train_mask]
    val_data   = data[val_mask]
    test_data  = data[test_mask]

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # # 1. Full data baseline
    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # # 2. Random 10%
    # print("\n=== RANDOM 10% ===")
    # results["random_10"] = run_experiment(
    #     "RAND10", random_compress(train_data, ratio=0.1), val_data, test_data, data
    # )

    # 3. Recency burst (decay from end) 10%
    print("\n=== RECENCY BURST (decay from end) 10% ===")
    results["recency_end_10"] = run_experiment(
        "REC_END",
        recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=True),
        val_data, test_data, data
    )

    # 4. Recency burst (decay from start) 10%
    print("\n=== RECENCY BURST (decay from start) 10% ===")
    results["recency_start_10"] = run_experiment(
        "REC_START",
        recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=False),
        val_data, test_data, data
    )

    # 5. Delta-t burst 10%
    print("\n=== DELTA-T BURST 10% ===")
    results["delta_t_10"] = run_experiment(
        "DELTA_T",
        delta_t_weighted_burst_compress(train_data, ratio=0.1),
        val_data, test_data, data
    )

    all_results[dataset_name] = results

print_results(all_results)

Using device: cuda

RUNNING DATASET: TGBL-UCI
Total events: 59835 | Train: 41884 | Val: 8975 | Test: 8976

=== RECENCY BURST (decay from end) 10% ===
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 27766 bursts in 41884 events.
   Merged to 27766. Sampling down to 4188...


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[REC_END] Ep 01 | Loss 2.8631 | Val AP 0.6558 | Test AP 0.6217 AUC 0.6855 MRR 0.9271
[REC_END] Ep 02 | Loss 4.5462 | Val AP 0.6585 | Test AP 0.6499 AUC 0.7058 MRR 0.8524
[REC_END] Ep 03 | Loss 4.0994 | Val AP 0.7058 | Test AP 0.8180 AUC 0.8359 MRR 0.9303
[REC_END] Ep 04 | Loss 2.1171 | Val AP 0.5805 | Test AP 0.5259 AUC 0.5478 MRR 0.9167
[REC_END] Ep 05 | Loss 3.2757 | Val AP 0.5766 | Test AP 0.5397 AUC 0.5829 MRR 0.7714
[REC_END] Ep 06 | Loss 1.8255 | Val AP 0.6503 | Test AP 0.6551 AUC 0.6980 MRR 0.8594
[REC_END] Ep 07 | Loss 1.7867 | Val AP 0.7215 | Test AP 0.7639 AUC 0.8055 MRR 0.9205
[REC_END] Ep 08 | Loss 1.6430 | Val AP 0.6990 | Test AP 0.7857 AUC 0.8084 MRR 0.9034
[REC_END] Ep 09 | Loss 1.9960 | Val AP 0.7088 | Test AP 0.7062 AUC 0.7684 MRR 0.9263
[REC_END] Ep 10 | Loss 2.2647 | Val AP 0.6921 | Test AP 0.6743 AUC 0.7416 MRR 0.9282
[REC_END] Ep 11 | Loss 2.5391 | Val AP 0.7021 | Test AP 0.7764 AUC 0.8169 MRR 0.9085
[REC_END] Ep 12 | Loss 2.6436 | Val AP 0.5801 | Test AP 0.5946 AU

/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[REC_START] Ep 01 | Loss 2.3002 | Val AP 0.5401 | Test AP 0.5001 AUC 0.5487 MRR 0.7734
[REC_START] Ep 02 | Loss 2.3536 | Val AP 0.5998 | Test AP 0.6497 AUC 0.7029 MRR 0.9041
[REC_START] Ep 03 | Loss 2.6593 | Val AP 0.6571 | Test AP 0.6701 AUC 0.7356 MRR 0.9173
[REC_START] Ep 04 | Loss 2.3076 | Val AP 0.5592 | Test AP 0.5149 AUC 0.4728 MRR 0.7618
[REC_START] Ep 05 | Loss 2.0394 | Val AP 0.6081 | Test AP 0.5876 AUC 0.6155 MRR 0.8067
[REC_START] Ep 06 | Loss 1.9755 | Val AP 0.7184 | Test AP 0.8113 AUC 0.8383 MRR 0.9296
[REC_START] Ep 07 | Loss 1.7079 | Val AP 0.6778 | Test AP 0.6167 AUC 0.6759 MRR 0.9243
[REC_START] Ep 08 | Loss 1.7404 | Val AP 0.6877 | Test AP 0.6490 AUC 0.7152 MRR 0.9301
[REC_START] Ep 09 | Loss 3.7613 | Val AP 0.6541 | Test AP 0.7401 AUC 0.7889 MRR 0.9227
[REC_START] Ep 10 | Loss 2.2508 | Val AP 0.6798 | Test AP 0.6691 AUC 0.7367 MRR 0.9290
[REC_START] Ep 11 | Loss 1.8303 | Val AP 0.5756 | Test AP 0.5199 AUC 0.3951 MRR 0.6746
[REC_START] Ep 12 | Loss 1.5590 | Val AP 0.

/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[DELTA_T] Ep 01 | Loss 2.7698 | Val AP 0.4497 | Test AP 0.3765 AUC 0.2246 MRR 0.6222
[DELTA_T] Ep 02 | Loss 3.7971 | Val AP 0.6895 | Test AP 0.7374 AUC 0.7948 MRR 0.9270
[DELTA_T] Ep 03 | Loss 2.4389 | Val AP 0.5701 | Test AP 0.5170 AUC 0.3547 MRR 0.6821
[DELTA_T] Ep 04 | Loss 1.8583 | Val AP 0.6381 | Test AP 0.7178 AUC 0.7599 MRR 0.9297
[DELTA_T] Ep 05 | Loss 3.7345 | Val AP 0.6905 | Test AP 0.7922 AUC 0.8035 MRR 0.8897
[DELTA_T] Ep 06 | Loss 1.7058 | Val AP 0.5315 | Test AP 0.5187 AUC 0.5256 MRR 0.7387
[DELTA_T] Ep 07 | Loss 2.5153 | Val AP 0.6825 | Test AP 0.6480 AUC 0.7158 MRR 0.9276
[DELTA_T] Ep 08 | Loss 1.5755 | Val AP 0.7562 | Test AP 0.8529 AUC 0.8691 MRR 0.9351
[DELTA_T] Ep 09 | Loss 2.4147 | Val AP 0.7478 | Test AP 0.8382 AUC 0.8623 MRR 0.9320
[DELTA_T] Ep 10 | Loss 2.5955 | Val AP 0.7341 | Test AP 0.6909 AUC 0.7525 MRR 0.9115
[DELTA_T] Ep 11 | Loss 2.4263 | Val AP 0.7030 | Test AP 0.6551 AUC 0.7272 MRR 0.9455
[DELTA_T] Ep 12 | Loss 4.0923 | Val AP 0.3759 | Test AP 0.3447 AU

ValueError: Unknown format code 'f' for object of type 'str'

In [1]:
import pandas as pd

In [13]:
# --- Imports ---
import os.path as osp
import torch
from torch_geometric.datasets import JODIEDataset

from utils import (
    device,
    run_experiment,
    random_compress,
    learned_compress,
    synthetic_burst_compress,
    subgraph_motif_compress,
    print_results,
)

print("Using device:", device)

# --------------------------------------------------
# Datasets to run
# --------------------------------------------------
DATASETS = ["wikipedia", "reddit", "mooc"]

all_results = {}

# --------------------------------------------------
# Loop over datasets
# --------------------------------------------------
for dataset_name in DATASETS:

    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    # --- Split ---
    train_data, val_data, test_data = data.train_val_test_split(
        val_ratio=0.15,
        test_ratio=0.15
    )

    # --- Move to device ---
    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # --------------------------------------------------
    # 1. Full data baseline
    # --------------------------------------------------
    print("\n=== FULL DATA ===")
    results["full"] = run_experiment(
        "FULL", train_data, val_data, test_data, data
    )

    # --------------------------------------------------
    # 2. Random 10% compression
    # --------------------------------------------------
    print("\n=== RANDOM 10% ===")
    results["random_10"] = run_experiment(
        "RAND10",
        random_compress(train_data, ratio=0.1),
        val_data,
        test_data,
        data,
    )

    # Save per-dataset results
    all_results[dataset_name] = results

# --------------------------------------------------
# Print final summary
# --------------------------------------------------
print_results(all_results)

Using device: cuda

RUNNING DATASET: MOOC


Processing...
Done!


Total events: 411749 | Train: 288224 | Val: 61762 | Test: 61763

=== FULL DATA ===


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[FULL] Ep 01 | Loss 0.7052 | Val AP 0.7789 | Test AP 0.7296 AUC 0.7715 MRR 0.8850
[FULL] Ep 02 | Loss 0.6436 | Val AP 0.8117 | Test AP 0.8038 AUC 0.8323 MRR 0.9137
[FULL] Ep 03 | Loss 0.6219 | Val AP 0.8225 | Test AP 0.7939 AUC 0.8304 MRR 0.9135
[FULL] Ep 04 | Loss 0.6172 | Val AP 0.8264 | Test AP 0.7846 AUC 0.8181 MRR 0.9075
[FULL] Ep 05 | Loss 0.6126 | Val AP 0.8455 | Test AP 0.8115 AUC 0.8429 MRR 0.9190
[FULL] Ep 06 | Loss 0.6049 | Val AP 0.8434 | Test AP 0.8255 AUC 0.8516 MRR 0.9241
[FULL] Ep 07 | Loss 0.5976 | Val AP 0.8437 | Test AP 0.8343 AUC 0.8639 MRR 0.9312
[FULL] Ep 08 | Loss 0.5921 | Val AP 0.8493 | Test AP 0.8394 AUC 0.8719 MRR 0.9353
[FULL] Ep 09 | Loss 0.5926 | Val AP 0.8469 | Test AP 0.8271 AUC 0.8589 MRR 0.9294
[FULL] Ep 10 | Loss 0.5817 | Val AP 0.8425 | Test AP 0.8251 AUC 0.8574 MRR 0.9277
[FULL] Ep 11 | Loss 0.5823 | Val AP 0.8574 | Test AP 0.8448 AUC 0.8747 MRR 0.9365
[FULL] Ep 12 | Loss 0.5935 | Val AP 0.8571 | Test AP 0.8461 AUC 0.8688 MRR 0.9320
[FULL] Ep 13 | L

/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[RAND10] Ep 01 | Loss 1.0314 | Val AP 0.5971 | Test AP 0.6050 AUC 0.6493 MRR 0.8280
[RAND10] Ep 02 | Loss 0.9141 | Val AP 0.6016 | Test AP 0.6085 AUC 0.6574 MRR 0.8314
[RAND10] Ep 03 | Loss 0.9125 | Val AP 0.6176 | Test AP 0.6253 AUC 0.6731 MRR 0.8368
[RAND10] Ep 04 | Loss 0.8903 | Val AP 0.6348 | Test AP 0.6308 AUC 0.6757 MRR 0.8348
[RAND10] Ep 05 | Loss 0.8912 | Val AP 0.6274 | Test AP 0.6256 AUC 0.6622 MRR 0.8261
[RAND10] Ep 06 | Loss 0.9120 | Val AP 0.6191 | Test AP 0.6133 AUC 0.6436 MRR 0.8223
[RAND10] Ep 07 | Loss 0.8905 | Val AP 0.6145 | Test AP 0.6193 AUC 0.6496 MRR 0.8246
[RAND10] Ep 08 | Loss 0.8872 | Val AP 0.6070 | Test AP 0.6074 AUC 0.6274 MRR 0.8129
[RAND10] Ep 09 | Loss 0.8883 | Val AP 0.6259 | Test AP 0.6263 AUC 0.6498 MRR 0.8240
[RAND10] Ep 10 | Loss 0.8745 | Val AP 0.6235 | Test AP 0.6158 AUC 0.6371 MRR 0.8184
[RAND10] Ep 11 | Loss 0.8748 | Val AP 0.6285 | Test AP 0.6341 AUC 0.6551 MRR 0.8266
[RAND10] Ep 12 | Loss 0.8598 | Val AP 0.6617 | Test AP 0.6305 AUC 0.6461 MRR

ValueError: not enough values to unpack (expected 3, got 2)

In [13]:
DATASETS = ["wikipedia", "reddit", "mooc"]

In [18]:
for dataset_name in DATASETS:

    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    # --- Split ---
    train_data, val_data, test_data = data.train_val_test_split(
        val_ratio=0.15,
        test_ratio=0.15
    )

    # --- Move to device ---
    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # --------------------------------------------------
    # 1. Full data baseline
    # --------------------------------------------------
    print("\n=== RECENCY BURST (decay from end) 10% ===")
    results["recency_end_10"] = run_experiment(
        "REC_END",
        recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=True),
        val_data, test_data, data
    )

    # 4. Recency burst (decay from start) 10%
    print("\n=== RECENCY BURST (decay from start) 10% ===")
    results["recency_start_10"] = run_experiment(
        "REC_START",
        recency_weighted_burst_compress(train_data, ratio=0.1, decay_from_end=False),
        val_data, test_data, data
    )

    # 5. Delta-t burst 10%
    print("\n=== DELTA-T BURST 10% ===")
    results["delta_t_10"] = run_experiment(
        "DELTA_T",
        delta_t_weighted_burst_compress(train_data, ratio=0.1),
        val_data, test_data, data
    )

    all_results[dataset_name] = results

    # Save per-dataset results
    all_results[dataset_name] = results


RUNNING DATASET: WIKIPEDIA
Total events: 157474 | Train: 110232 | Val: 23621 | Test: 23621

=== RECENCY BURST (decay from end) 10% ===
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 45221 bursts in 110232 events.
   Merged to 45221. Sampling down to 11023...


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[REC_END] Ep 01 | Loss 1.3094 | Val AP 0.7293 | Test AP 0.6961 AUC 0.7292 MRR 0.8748
[REC_END] Ep 02 | Loss 1.2210 | Val AP 0.8191 | Test AP 0.7543 AUC 0.7743 MRR 0.8962
[REC_END] Ep 03 | Loss 1.1227 | Val AP 0.8891 | Test AP 0.8516 AUC 0.8676 MRR 0.9328
[REC_END] Ep 04 | Loss 1.0957 | Val AP 0.9109 | Test AP 0.8970 AUC 0.8931 MRR 0.9422
[REC_END] Ep 05 | Loss 1.0557 | Val AP 0.9114 | Test AP 0.9080 AUC 0.9031 MRR 0.9477
[REC_END] Ep 06 | Loss 1.0384 | Val AP 0.9223 | Test AP 0.9133 AUC 0.9096 MRR 0.9501
[REC_END] Ep 07 | Loss 1.0238 | Val AP 0.9099 | Test AP 0.8914 AUC 0.8915 MRR 0.9404
[REC_END] Ep 08 | Loss 1.0035 | Val AP 0.9351 | Test AP 0.9270 AUC 0.9213 MRR 0.9536
[REC_END] Ep 09 | Loss 0.9997 | Val AP 0.9304 | Test AP 0.9232 AUC 0.9189 MRR 0.9509
[REC_END] Ep 10 | Loss 0.9876 | Val AP 0.9388 | Test AP 0.9282 AUC 0.9213 MRR 0.9526
[REC_END] Ep 11 | Loss 0.9730 | Val AP 0.9338 | Test AP 0.9275 AUC 0.9218 MRR 0.9533
[REC_END] Ep 12 | Loss 0.9624 | Val AP 0.9414 | Test AP 0.9296 AU

KeyboardInterrupt: 

In [26]:
# --- Imports ---
import os
import os.path as osp
import torch
from torch_geometric.datasets import JODIEDataset

from utils import (
    device,
    random_compress,
)

print("Using device:", device)

# --------------------------------------------------
# Datasets to run
# --------------------------------------------------
DATASETS = ["wikipedia", "reddit", "mooc"]

# Create directory
os.makedirs("compressed_jodie", exist_ok=True)

# --------------------------------------------------
# Loop over datasets
# --------------------------------------------------
for dataset_name in DATASETS:

    print(f"\n{'='*80}")
    print(f"PROCESSING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    # --- Split ---
    train_data, _, _ = data.train_val_test_split(
        val_ratio=0.15,
        test_ratio=0.15
    )

    train_data = train_data.to(device)

    print(f"Train events: {train_data.num_events}")

    # ==================================================
    # RANDOM 10%
    # ==================================================
    rand_path = f"compressed_jodie/{dataset_name}_random_10.pt"

    if not os.path.exists(rand_path):
        print("Computing RANDOM 10%...")
        rand_data = random_compress(train_data, ratio=0.1)
        torch.save(rand_data.cpu(), rand_path)
    else:
        print("RANDOM 10% already exists, skipping...")

    # ==================================================
    # RECENCY END
    # ==================================================
    rec_end_path = f"compressed_jodie/{dataset_name}_recency_end_10.pt"

    if not os.path.exists(rec_end_path):
        print("Computing RECENCY END...")
        rec_end = recency_weighted_burst_compress(
            train_data, ratio=0.1, decay_from_end=True
        )
        torch.save(rec_end.cpu(), rec_end_path)
    else:
        print("RECENCY END already exists, skipping...")

    # ==================================================
    # RECENCY START
    # ==================================================
    rec_start_path = f"compressed_jodie/{dataset_name}_recency_start_10.pt"

    if not os.path.exists(rec_start_path):
        print("Computing RECENCY START...")
        rec_start = recency_weighted_burst_compress(
            train_data, ratio=0.1, decay_from_end=False
        )
        torch.save(rec_start.cpu(), rec_start_path)
    else:
        print("RECENCY START already exists, skipping...")

    # ==================================================
    # DELTA-T
    # ==================================================
    delta_path = f"compressed_jodie/{dataset_name}_delta_t_10.pt"

    if not os.path.exists(delta_path):
        print("Computing DELTA-T...")
        delta_t = delta_t_weighted_burst_compress(train_data, ratio=0.1)
        torch.save(delta_t.cpu(), delta_path)
    else:
        print("DELTA-T already exists, skipping...")

print("\n✅ All compressions saved!")

Using device: cuda

PROCESSING DATASET: WIKIPEDIA
Train events: 110232
RANDOM 10% already exists, skipping...
Computing RECENCY END...
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 45221 bursts in 110232 events.
   Merged to 45221. Sampling down to 11023...
Computing RECENCY START...
--- Recency-Weighted Burst Compress (decay_from_end=False) ---
   Found 45221 bursts in 110232 events.
   Merged to 45221. Sampling down to 11023...
Computing DELTA-T...
--- Delta-t Weighted Burst Compress ---
   Found 45221 bursts in 110232 events.
   Merged to 45221. Sampling down to 11023...

PROCESSING DATASET: REDDIT


Processing...
Done!


Train events: 470713
Computing RANDOM 10%...
Computing RECENCY END...
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 383953 bursts in 470713 events.
   Merged to 383953. Sampling down to 47071...
Computing RECENCY START...
--- Recency-Weighted Burst Compress (decay_from_end=False) ---
   Found 383953 bursts in 470713 events.
   Merged to 383953. Sampling down to 47071...
Computing DELTA-T...
--- Delta-t Weighted Burst Compress ---
   Found 383953 bursts in 470713 events.
   Merged to 383953. Sampling down to 47071...

PROCESSING DATASET: MOOC


Processing...
Done!


Train events: 288224
Computing RANDOM 10%...
Computing RECENCY END...
--- Recency-Weighted Burst Compress (decay_from_end=True) ---
   Found 194951 bursts in 288224 events.
   Merged to 194951. Sampling down to 28822...
Computing RECENCY START...
--- Recency-Weighted Burst Compress (decay_from_end=False) ---
   Found 194951 bursts in 288224 events.
   Merged to 194951. Sampling down to 28822...
Computing DELTA-T...
--- Delta-t Weighted Burst Compress ---
   Found 194951 bursts in 288224 events.
   Merged to 194951. Sampling down to 28822...

✅ All compressions saved!


In [ ]:
# Adaptive mean window
print("\n=== ADAPTIVE RECENCY BURST (mean, decay_from_end) 10% ===")
results["adaptive_rec_end_mean_10"] = run_experiment(
    "ADAP_REC_END_MEAN",
    adaptive_recency_burst_compress(train_data, ratio=0.1, stat="mean", decay_from_end=True),
    val_data, test_data, data, 100
)

print("\n=== ADAPTIVE RECENCY BURST (median, decay_from_end) 10% ===")
results["adaptive_rec_end_median_10"] = run_experiment(
    "ADAP_REC_END_MED",
    adaptive_recency_burst_compress(train_data, ratio=0.1, stat="median", decay_from_end=True),
    val_data, test_data, data, 100
)

# Adaptive mean/median for delta-t variant too
print("\n=== ADAPTIVE DELTA-T BURST (mean) 10% ===")
results["adaptive_delta_t_mean_10"] = run_experiment(
    "ADAP_DELTA_MEAN",
    adaptive_delta_t_burst_compress(train_data, ratio=0.1, stat="mean"),
    val_data, test_data, data, 100
)

print("\n=== ADAPTIVE DELTA-T BURST (median) 10% ===")
results["adaptive_delta_t_median_10"] = run_experiment(
    "ADAP_DELTA_MED",
    adaptive_delta_t_burst_compress(train_data, ratio=0.1, stat="median"),
    val_data, test_data, data, 100
)

In [28]:
def binary_search_window_compress(td, ratio=0.1, tol=0.02, seed=42,
                                   w_lo=1.0, w_hi=1e7, max_iter=30):
    """
    Per-(src,dst)-pair binary search over time window size to hit
    target compression ratio in [ratio-tol, ratio+tol].

    For each pair:
      - Binary search finds window w* such that ceil(pair_events / w*-bucket-count)
        lands in [target_lo, target_hi] events for that pair.
      - Events in each window are aggregated via delta-t weighted mean (same as
        delta_t_weighted_burst_compress).
      - Representative timestamp = last event in window.

    Global result is concatenated across all pairs, then time-sorted.
    """
    print("--- Binary Search Window Compress ---")
    t_dtype   = td.t.dtype
    msg_dtype = td.msg.dtype
    dev       = td.msg.device

    df = pd.DataFrame({
        "src": td.src.cpu().numpy(),
        "dst": td.dst.cpu().numpy(),
        "t":   td.t.cpu().numpy(),
        "idx": np.arange(len(td.src)),
    }).sort_values(["src", "dst", "t"]).reset_index(drop=True)

    msg_np  = td.msg.cpu().float().numpy()      # [N, D]
    msg_dim = td.msg.size(1)
    total_N = len(df)

    target_lo = ratio - tol   # 0.08
    target_hi = ratio + tol   # 0.12

    all_src, all_dst, all_t, all_msg = [], [], [], []

    pairs = df.groupby(["src", "dst"], sort=False)

    for (src_id, dst_id), grp in pairs:
        grp      = grp.reset_index(drop=True)
        n_pair   = len(grp)
        t_vals   = grp["t"].values.astype(np.float64)
        idxs     = grp["idx"].values

        target_n_lo = max(1, int(np.floor(n_pair * target_lo)))
        target_n_hi = max(1, int(np.ceil(n_pair * target_hi)))

        # ── helper: compress pair with a fixed window size ──────────────
        def compress_with_window(w):
            """Returns (n_compressed, src_list, dst_list, t_list, msg_list)."""
            # assign each event to a bucket: floor(t / w)
            buckets   = np.floor(t_vals / w).astype(np.int64)
            _, inv, _ = np.unique(buckets, return_inverse=True,
                                  return_counts=True)

            n_buckets = int(inv.max()) + 1
            msgs      = msg_np[idxs]           # [n_pair, D]

            # delta-t weights within pair (same logic as strategy 1)
            dt        = np.diff(t_vals, prepend=t_vals[0])
            dt[0]     = 0.0
            weights   = np.log1p(dt).clip(min=1.0)[:, None]  # [n_pair, 1]

            # weighted sum per bucket
            w_msg  = np.zeros((n_buckets, msg_dim), dtype=np.float32)
            w_sum  = np.zeros((n_buckets, 1),       dtype=np.float32)
            t_rep  = np.zeros(n_buckets,             dtype=np.float64)

            for i in range(n_pair):
                b = inv[i]
                w_msg[b] += weights[i] * msgs[i]
                w_sum[b] += weights[i]
                t_rep[b]  = t_vals[i]           # last-event timestamp per bucket

            agg_msg = w_msg / w_sum.clip(min=1e-8)
            return n_buckets, agg_msg, t_rep

        # ── binary search over w ─────────────────────────────────────────
        # larger w  → fewer buckets → more compression (fewer output events)
        # smaller w → more buckets  → less compression (closer to original)

        lo, hi   = float(w_lo), float(w_hi)
        best_w   = hi          # default: maximum compression
        converged = False

        for _ in range(max_iter):
            mid      = (lo + hi) / 2.0
            n_out, _, _ = compress_with_window(mid)

            if target_n_lo <= n_out <= target_n_hi:
                best_w    = mid
                converged = True
                break
            elif n_out > target_n_hi:
                # too many events → need larger window
                lo = mid
            else:
                # too few events → need smaller window
                hi = mid

            if hi - lo < 1e-3:          # numerical convergence
                best_w = mid
                break

        if not converged and n_pair == 1:
            best_w = w_lo               # single-event pair: keep as-is

        n_out, agg_msg, t_rep = compress_with_window(best_w)

        all_src.append(np.full(n_out, src_id,  dtype=np.int64))
        all_dst.append(np.full(n_out, dst_id,  dtype=np.int64))
        all_t.append(t_rep)
        all_msg.append(agg_msg)

    # ── concatenate & sort by time ───────────────────────────────────────
    all_src = np.concatenate(all_src)
    all_dst = np.concatenate(all_dst)
    all_t   = np.concatenate(all_t)
    all_msg = np.concatenate(all_msg, axis=0)

    order   = np.argsort(all_t, kind="stable")
    all_src = all_src[order]
    all_dst = all_dst[order]
    all_t   = all_t[order]
    all_msg = all_msg[order]

    actual_ratio = len(all_src) / total_N
    print(f"   Input: {total_N} events | Output: {len(all_src)} events "
          f"| Actual ratio: {actual_ratio:.4f} "
          f"(target: {target_lo:.2f}–{target_hi:.2f})")

    return TemporalData(
        src = torch.tensor(all_src, dtype=torch.long),
        dst = torch.tensor(all_dst, dtype=torch.long),
        t   = torch.tensor(all_t,   dtype=t_dtype),
        msg = torch.tensor(all_msg, dtype=msg_dtype),
    )

In [ ]:
# --- Imports ---
import os.path as osp
import torch
import pandas as pd
from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset
from torch_geometric.datasets import JODIEDataset
from utils import (
    device,
    run_experiment,
    print_results,
)
print("Using device:", device)

all_results = {}

# --------------------------------------------------
# 1. TGB datasets (UCI, Flight)
# --------------------------------------------------
TGB_DATASETS = ["tgbl-uci"]

for dataset_name in TGB_DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
    data = dataset.get_TemporalData()

    train_data = data[dataset.train_mask]
    val_data   = data[dataset.val_mask]
    test_data  = data[dataset.test_mask]

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(f"Total: {data.num_events} | Train: {train_data.num_events} | Val: {val_data.num_events} | Test: {test_data.num_events}")

    results = {}

    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # print("\n=== RANDOM 10% ===")
    # results["random_10"] = run_experiment(
    #     "RAND10", random_compress(train_data, ratio=0.1), val_data, test_data, data
    # )

    print("\n=== BINARY SEARCH WINDOW 10% ===")
    results["binary_search_10"] = run_experiment(
        "BSWIN10", binary_search_window_compress(train_data, ratio=0.1), val_data, test_data, data
    )

    all_results[dataset_name] = results

# --------------------------------------------------
# 2. JODIE datasets (Wikipedia, Reddit, MOOC)
# --------------------------------------------------
JODIE_DATASETS = ["wikipedia", "reddit", "mooc"]

for dataset_name in JODIE_DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(f"Total: {data.num_events} | Train: {train_data.num_events} | Val: {val_data.num_events} | Test: {test_data.num_events}")

    results = {}

    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # print("\n=== RANDOM 10% ===")
    # results["random_10"] = run_experiment(
    #     "RAND10", random_compress(train_data, ratio=0.1), val_data, test_data, data
    # )

    print("\n=== BINARY SEARCH WINDOW 10% ===")
    results["binary_search_10"] = run_experiment(
        "BSWIN10", binary_search_window_compress(train_data, ratio=0.1), val_data, test_data, data
    )

    all_results[dataset_name] = results

# --------------------------------------------------
# Save to CSV
# --------------------------------------------------
rows = []
for dataset_name, results in all_results.items():
    for method, metrics in results.items():
        rows.append({
            "dataset": dataset_name,
            "method":  method,
            **metrics,   # assumes run_experiment returns a dict of metric_name: value
        })

df = pd.DataFrame(rows)
df.to_csv("results_binary_search.csv", index=False)
print("\nResults saved to results_binary_search.csv")
print(df.to_string(index=False))

Using device: cuda

RUNNING DATASET: TGBL-UCI
Total: 59835 | Train: 41884 | Val: 8975 | Test: 8976

=== BINARY SEARCH WINDOW 10% ===
--- Binary Search Window Compress ---
   Input: 41884 events | Output: 14611 events | Actual ratio: 0.3488 (target: 0.08–0.12)


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[BSWIN10] Ep 01 | Loss 12.8290 | Val AP 0.7272 | Test AP 0.8174 AUC 0.8642 MRR 0.9457
[BSWIN10] Ep 02 | Loss 5.8198 | Val AP 0.7094 | Test AP 0.7389 AUC 0.7929 MRR 0.9325
[BSWIN10] Ep 03 | Loss 5.7104 | Val AP 0.7305 | Test AP 0.8294 AUC 0.8622 MRR 0.9433
[BSWIN10] Ep 04 | Loss 6.8742 | Val AP 0.6561 | Test AP 0.6790 AUC 0.7523 MRR 0.9402
[BSWIN10] Ep 05 | Loss 5.2011 | Val AP 0.7262 | Test AP 0.8221 AUC 0.8576 MRR 0.9339
[BSWIN10] Ep 06 | Loss 8.0721 | Val AP 0.5431 | Test AP 0.5469 AUC 0.6038 MRR 0.8107
[BSWIN10] Ep 07 | Loss 5.5509 | Val AP 0.5999 | Test AP 0.6340 AUC 0.7453 MRR 0.8839
[BSWIN10] Ep 08 | Loss 4.4040 | Val AP 0.6108 | Test AP 0.6108 AUC 0.6746 MRR 0.9460
[BSWIN10] Ep 09 | Loss 5.8902 | Val AP 0.7680 | Test AP 0.7767 AUC 0.8405 MRR 0.9501
[BSWIN10] Ep 10 | Loss 3.5099 | Val AP 0.5330 | Test AP 0.5528 AUC 0.6312 MRR 0.8521
[BSWIN10] Ep 11 | Loss 7.0396 | Val AP 0.6376 | Test AP 0.6517 AUC 0.7266 MRR 0.9493
[BSWIN10] Ep 12 | Loss 4.0052 | Val AP 0.6135 | Test AP 0.6197 A

/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[BSWIN10] Ep 01 | Loss 1.3228 | Val AP 0.5918 | Test AP 0.6050 AUC 0.6353 MRR 0.8202
[BSWIN10] Ep 02 | Loss 1.2370 | Val AP 0.8618 | Test AP 0.8410 AUC 0.8592 MRR 0.9271
[BSWIN10] Ep 03 | Loss 1.1616 | Val AP 0.9161 | Test AP 0.8901 AUC 0.8910 MRR 0.9479
[BSWIN10] Ep 04 | Loss 1.1025 | Val AP 0.9150 | Test AP 0.8967 AUC 0.9127 MRR 0.9551
[BSWIN10] Ep 05 | Loss 1.1030 | Val AP 0.9305 | Test AP 0.9259 AUC 0.9266 MRR 0.9581
[BSWIN10] Ep 06 | Loss 1.0912 | Val AP 0.9237 | Test AP 0.9147 AUC 0.9174 MRR 0.9577
[BSWIN10] Ep 07 | Loss 1.0714 | Val AP 0.9233 | Test AP 0.8999 AUC 0.9048 MRR 0.9499
[BSWIN10] Ep 08 | Loss 1.0575 | Val AP 0.9372 | Test AP 0.9176 AUC 0.9185 MRR 0.9525
[BSWIN10] Ep 09 | Loss 1.0505 | Val AP 0.9326 | Test AP 0.9155 AUC 0.9170 MRR 0.9545
[BSWIN10] Ep 10 | Loss 1.0291 | Val AP 0.9323 | Test AP 0.9102 AUC 0.9119 MRR 0.9500
[BSWIN10] Ep 11 | Loss 1.0279 | Val AP 0.9380 | Test AP 0.9237 AUC 0.9257 MRR 0.9552
[BSWIN10] Ep 12 | Loss 1.0148 | Val AP 0.9398 | Test AP 0.9233 AU

Processing...
Done!


Total: 672447 | Train: 470713 | Val: 100867 | Test: 100867

=== BINARY SEARCH WINDOW 10% ===
--- Binary Search Window Compress ---
   Input: 470713 events | Output: 81318 events | Actual ratio: 0.1728 (target: 0.08–0.12)


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[BSWIN10] Ep 01 | Loss 0.9447 | Val AP 0.9370 | Test AP 0.9276 AUC 0.9352 MRR 0.9595
[BSWIN10] Ep 02 | Loss 0.8670 | Val AP 0.9618 | Test AP 0.9574 AUC 0.9601 MRR 0.9717
[BSWIN10] Ep 03 | Loss 0.8383 | Val AP 0.9678 | Test AP 0.9621 AUC 0.9644 MRR 0.9768
[BSWIN10] Ep 04 | Loss 0.8242 | Val AP 0.9623 | Test AP 0.9526 AUC 0.9543 MRR 0.9724
[BSWIN10] Ep 05 | Loss 0.8189 | Val AP 0.9703 | Test AP 0.9663 AUC 0.9681 MRR 0.9790
[BSWIN10] Ep 06 | Loss 0.7972 | Val AP 0.9657 | Test AP 0.9517 AUC 0.9581 MRR 0.9741
[BSWIN10] Ep 07 | Loss 0.7970 | Val AP 0.9691 | Test AP 0.9628 AUC 0.9654 MRR 0.9770
[BSWIN10] Ep 08 | Loss 0.7861 | Val AP 0.9731 | Test AP 0.9717 AUC 0.9727 MRR 0.9809
[BSWIN10] Ep 09 | Loss 0.7777 | Val AP 0.9737 | Test AP 0.9690 AUC 0.9711 MRR 0.9807
[BSWIN10] Ep 10 | Loss 0.7769 | Val AP 0.9729 | Test AP 0.9690 AUC 0.9703 MRR 0.9796
[BSWIN10] Ep 11 | Loss 0.7732 | Val AP 0.9705 | Test AP 0.9646 AUC 0.9651 MRR 0.9769
[BSWIN10] Ep 12 | Loss 0.7678 | Val AP 0.9702 | Test AP 0.9662 AU

In [ ]:
JODIE_DATASETS = ["mooc"]

for dataset_name in JODIE_DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)

    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(f"Total: {data.num_events} | Train: {train_data.num_events} | Val: {val_data.num_events} | Test: {test_data.num_events}")

    results = {}

    # print("\n=== FULL DATA ===")
    # results["full"] = run_experiment("FULL", train_data, val_data, test_data, data)

    # print("\n=== RANDOM 10% ===")
    # results["random_10"] = run_experiment(
    #     "RAND10", random_compress(train_data, ratio=0.1), val_data, test_data, data
    # )

    print("\n=== BINARY SEARCH WINDOW 10% ===")
    results["binary_search_10"] = run_experiment(
        "BSWIN10", binary_search_window_compress(train_data, ratio=0.1), val_data, test_data, data
    )

    all_results[dataset_name] = results

# --------------------------------------------------
# Save to CSV
# --------------------------------------------------
rows = []
for dataset_name, results in all_results.items():
    for method, metrics in results.items():
        rows.append({
            "dataset": dataset_name,
            "method":  method,
            **metrics,   # assumes run_experiment returns a dict of metric_name: value
        })

df = pd.DataFrame(rows)
df.to_csv("results_binary_search.csv", index=False)
print("\nResults saved to results_binary_search.csv")
print(df.to_string(index=False))


RUNNING DATASET: MOOC
Total: 411749 | Train: 288224 | Val: 61762 | Test: 61763

=== BINARY SEARCH WINDOW 10% ===
--- Binary Search Window Compress ---
   Input: 288224 events | Output: 128726 events | Actual ratio: 0.4466 (target: 0.08–0.12)


/usr/local/lib/python3.12/dist-packages/torch/_compile.py:53: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[BSWIN10] Ep 01 | Loss 0.8336 | Val AP 0.7605 | Test AP 0.7657 AUC 0.8106 MRR 0.9053
[BSWIN10] Ep 02 | Loss 0.7702 | Val AP 0.8028 | Test AP 0.7956 AUC 0.8402 MRR 0.9203
[BSWIN10] Ep 03 | Loss 0.7428 | Val AP 0.8158 | Test AP 0.8069 AUC 0.8442 MRR 0.9211
[BSWIN10] Ep 04 | Loss 0.7151 | Val AP 0.8134 | Test AP 0.8041 AUC 0.8408 MRR 0.9176
[BSWIN10] Ep 05 | Loss 0.7077 | Val AP 0.8224 | Test AP 0.8020 AUC 0.8312 MRR 0.9091
[BSWIN10] Ep 06 | Loss 0.7039 | Val AP 0.8179 | Test AP 0.7880 AUC 0.8221 MRR 0.9076
[BSWIN10] Ep 07 | Loss 0.6994 | Val AP 0.8202 | Test AP 0.7991 AUC 0.8335 MRR 0.9134
[BSWIN10] Ep 08 | Loss 0.6997 | Val AP 0.8050 | Test AP 0.7548 AUC 0.7933 MRR 0.8904
[BSWIN10] Ep 09 | Loss 0.6939 | Val AP 0.8220 | Test AP 0.7926 AUC 0.8235 MRR 0.9104
[BSWIN10] Ep 10 | Loss 0.6926 | Val AP 0.8085 | Test AP 0.7797 AUC 0.8124 MRR 0.9008
[BSWIN10] Ep 11 | Loss 0.6829 | Val AP 0.8288 | Test AP 0.8081 AUC 0.8389 MRR 0.9167
[BSWIN10] Ep 12 | Loss 0.6793 | Val AP 0.8170 | Test AP 0.7888 AU

In [32]:
# --- Imports ---
import os
import os.path as osp
import torch
import pandas as pd
import zipfile

from torch_geometric.datasets import JODIEDataset

from utils import (
    device,
    run_experiment,
)

print("Using device:", device)

# --------------------------------------------------
# Config
# --------------------------------------------------
JODIE_DATASETS = ["wikipedia", "reddit", "mooc", "tgbl-uci"]
# COMPRESS_DIR = "compressed_jodie_bs"

# os.makedirs(COMPRESS_DIR, exist_ok=True)

all_results = {}

# --------------------------------------------------
# Loop over datasets
# --------------------------------------------------
for dataset_name in JODIE_DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    path = osp.join("data", "JODIE", dataset_name)
    dataset = JODIEDataset(path, name=dataset_name)
    data = dataset[0]

    # --- Split ---
    train_data, val_data, test_data = data.train_val_test_split(
        val_ratio=0.15,
        test_ratio=0.15
    )

    # --- Move to device ---
    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # ==================================================
    # BINARY SEARCH WINDOW 10% (with caching)
    # ==================================================
    # ==================================================
    # BINARY SEARCH WINDOW 10% (ALWAYS RECOMPUTE)
    # ==================================================
    bs_path = f"{COMPRESS_DIR}/{dataset_name}_binary_search_10.pt"
    
    print("\nComputing BINARY SEARCH 10% (overwrite mode)...")
    bs_data = binary_search_window_compress(train_data, ratio=0.1)
    
    # Always overwrite
    torch.save(bs_data.cpu(), bs_path)
    
    print(f"Saved to {bs_path}")
    # results["binary_search_10"] = run_experiment(
    #     "BSWIN10",
    #     bs_data,
    #     val_data,
    #     test_data,
    #     data
    # )

    # all_results[dataset_name] = results


# --------------------------------------------------
# Save results to CSV
# --------------------------------------------------
rows = []
for dataset_name, results in all_results.items():
    for method, metrics in results.items():
        rows.append({
            "dataset": dataset_name,
            "method": method,
            **metrics,
        })

df = pd.DataFrame(rows)
csv_path = "results_binary_search.csv"
df.to_csv(csv_path, index=False)

print(f"\nResults saved to {csv_path}")
print(df.to_string(index=False))


# --------------------------------------------------
# ZIP the compressed folder
# --------------------------------------------------
zip_path = f"{COMPRESS_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(COMPRESS_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, COMPRESS_DIR)
            zipf.write(file_path, arcname)

print(f"\n✅ Compressed folder zipped at: {zip_path}")

Using device: cuda

RUNNING DATASET: WIKIPEDIA
Total: 157474 | Train: 110232 | Val: 23621 | Test: 23621

Computing BINARY SEARCH 10% (overwrite mode)...
--- Binary Search Window Compress ---
   Input: 110232 events | Output: 18886 events | Actual ratio: 0.1713 (target: 0.08–0.12)
Saved to compressed_jodie_bs/wikipedia_binary_search_10.pt

RUNNING DATASET: REDDIT
Total: 672447 | Train: 470713 | Val: 100867 | Test: 100867

Computing BINARY SEARCH 10% (overwrite mode)...
--- Binary Search Window Compress ---
   Input: 470713 events | Output: 81318 events | Actual ratio: 0.1728 (target: 0.08–0.12)
Saved to compressed_jodie_bs/reddit_binary_search_10.pt

RUNNING DATASET: MOOC
Total: 411749 | Train: 288224 | Val: 61762 | Test: 61763

Computing BINARY SEARCH 10% (overwrite mode)...
--- Binary Search Window Compress ---
   Input: 288224 events | Output: 128726 events | Actual ratio: 0.4466 (target: 0.08–0.12)
Saved to compressed_jodie_bs/mooc_binary_search_10.pt

RUNNING DATASET: TGBL-UCI


AssertionError: 

In [33]:
# --- Imports ---
import os
import torch
import pandas as pd
import zipfile

from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset

from utils import (
    device,
    run_experiment,
)

print("Using device:", device)

# --------------------------------------------------
# Config
# --------------------------------------------------
DATASETS = ["tgbl-uci"]
COMPRESS_DIR = "compressed_tgbl_bs"

os.makedirs(COMPRESS_DIR, exist_ok=True)

all_results = {}

# --------------------------------------------------
# Loop over datasets
# --------------------------------------------------
for dataset_name in DATASETS:
    print(f"\n{'='*80}")
    print(f"RUNNING DATASET: {dataset_name.upper()}")
    print(f"{'='*80}")

    # --- Load Dataset ---
    dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
    data = dataset.get_TemporalData()

    train_mask = dataset.train_mask
    val_mask   = dataset.val_mask
    test_mask  = dataset.test_mask

    train_data = data[train_mask]
    val_data   = data[val_mask]
    test_data  = data[test_mask]

    # --- Move to device ---
    data       = data.to(device)
    train_data = train_data.to(device)
    val_data   = val_data.to(device)
    test_data  = test_data.to(device)

    print(
        f"Total events: {data.num_events} | "
        f"Train: {train_data.num_events} | "
        f"Val: {val_data.num_events} | "
        f"Test: {test_data.num_events}"
    )

    results = {}

    # ==================================================
    # BINARY SEARCH WINDOW 10% (ALWAYS OVERWRITE)
    # ==================================================
    bs_path = f"{COMPRESS_DIR}/{dataset_name}_binary_search_10.pt"

    print("\nComputing BINARY SEARCH 10% (overwrite mode)...")
    bs_data = binary_search_window_compress(train_data, ratio=0.1)

    torch.save(bs_data.cpu(), bs_path)
    print(f"Saved to {bs_path}")

    # --------------------------------------------------
    # Run experiment
    # --------------------------------------------------
    # print("\n=== BINARY SEARCH WINDOW 10% ===")
    # results["binary_search_10"] = run_experiment(
    #     "BSWIN10",
    #     bs_data,
    #     val_data,
    #     test_data,
    #     data
    # )

    # all_results[dataset_name] = results


# --------------------------------------------------
# Save results to CSV
# --------------------------------------------------
rows = []
for dataset_name, results in all_results.items():
    for method, metrics in results.items():
        rows.append({
            "dataset": dataset_name,
            "method": method,
            **metrics,
        })

df = pd.DataFrame(rows)
csv_path = "results_tgbl_binary_search.csv"
df.to_csv(csv_path, index=False)

print(f"\nResults saved to {csv_path}")
print(df.to_string(index=False))


# --------------------------------------------------
# ZIP the compressed folder
# --------------------------------------------------
zip_path = f"{COMPRESS_DIR}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(COMPRESS_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, COMPRESS_DIR)
            zipf.write(file_path, arcname)

print(f"\n✅ Compressed folder zipped at: {zip_path}")

Using device: cuda

RUNNING DATASET: TGBL-UCI
Total events: 59835 | Train: 41884 | Val: 8975 | Test: 8976

Computing BINARY SEARCH 10% (overwrite mode)...
--- Binary Search Window Compress ---
   Input: 41884 events | Output: 14611 events | Actual ratio: 0.3488 (target: 0.08–0.12)
Saved to compressed_tgbl_bs/tgbl-uci_binary_search_10.pt

Results saved to results_tgbl_binary_search.csv
Empty DataFrame
Columns: []
Index: []

✅ Compressed folder zipped at: compressed_tgbl_bs.zip


In [34]:
# --------------------------------------------------
# ZIP the compressed_jodie folder
# --------------------------------------------------
import zipfile
import os

folder_to_zip = "compressed_jodie"
zip_path = f"{folder_to_zip}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(folder_to_zip):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, folder_to_zip)
            zipf.write(file_path, arcname)

print(f"\n✅ compressed_jodie folder zipped at: {zip_path}")


✅ compressed_jodie folder zipped at: compressed_jodie.zip
